한 번 api를 이용해 메시지를 전달하고 답변을 받아보겠습니다!

우선 필요한 라이브러리들을 모두 불러오고, 환경 값들을 불러올거에요.

In [ ]:
# api 호출을 위해 필요한 langchain의 라이브러리들 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 파일 관리와 .env 파일을 관리하기 위한 라이브러리.
# 파이썬 기본 라이브러리로 따로 설치할 필요가 없다.
import os
from dotenv import load_dotenv, find_dotenv

# .env 파일로부터 api_key를 불러오는 코드
# .env 파일은 최상위 폴더에 존재한다.
load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

모델에서 설정 가능한 설정값을 소개하겠습니다.

* temperature : 0~2 사이의 숫자로 클수록 더 다양한 답변을 생성하게 됨.
* max_tokens : 모델이 생성 가능한 최대 토큰 수 설정.
* timeout : 지정한 시간 이상 응답이 없으면 중지.
* max_retries : 요청이 실패했을 때 재시도 횟수.

In [ ]:
# 모델 설정
model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
    max_tokesn=1000
    timeout=30, # 초
    max_retries=3
)

모델에 메세지는 아래와 같이 입력할 수 있습니다. 모델의 파라미터를 마음대로 조절하면서 답변이 어떻게 바뀌는지 확인해 봅시다.

In [5]:
user_message = "안녕, 내 이름은 굳건이야. 앞으로 잘 부탁해."

result = model.invoke(user_message)

print(result.content)

안녕하세요, 굳건이님! 반갑습니다. 앞으로 잘 부탁드려요.  
편하게 말씀해 주세요 — 필요한 거 있으면 바로 도와드릴게요.


메시지는 리스트를 이용해 여러 개의 메시지를 담을 수 있습니다.

각 메시지에 담기는 내용은 dictionary 형태로 작성할 수 있어요.
* "role" : 메세지의 주체를 설정.
    * "system" : 시스템 프롬프트
    * "user" : 사용자가 작성하는 메시지
    * "assistant" : LLM이 답변한 메시지.
* "content" : 메시지의 내용

```
messages = [
    {
        "role": "system",
        "content": "너는 경비아저씨야."
    },
    {
        "role": "user",
        "content": "안녕하세요."
    }
]
```

이렇게 "role"을 어떻게 설정하냐에 따라 모델이 메세지가 누가 보낸건지를 판별할 수 있습니다.

시스템 프롬프트를 바꿔 보면서 확인해 봅시다.

In [6]:
messages = [
    {"role": "system", "content": "애교 섞인 말투로 대답해."}
]

user_message = "안녕, 내 이름은 굳건이야. 앞으로 잘 부탁해."
messages.append(
    {"role": "user", "content": user_message}
)

result = model.invoke(messages)

print(result.content)

안녕, 굳건이! 반가워요 😊  
앞으로 잘 부탁해요. 내가 든든하게 도와줄게요!


`ChatPromptTemplate`은 반복되는 프롬프트를 패턴화 할 수 있습니다.

`StrOutputParser`는 모델의 출력 결과에서 텍스트 메세지만 추출하는 역할을 합니다.

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 비즈니스 이메일 작성 전문가입니다. {tone} 톤으로 작성하세요."),
    ("human", """
    수신자: {recipient}
    목적: {purpose}
    핵심 내용: {key_points}

    위 정보를 바탕으로 이메일을 작성해주세요.
    """),
])

chain = prompt | model | StrOutputParser()

response = chain.invoke({
    "tone": "정중하고 전문적인",
    "recipient": "김철수 팀장님",
    "purpose": "프로젝트 일정 변경 요청",
    "key_points": "개발 일정 2주 연장, 이유는 요구사항 추가",
})

print(response)

제목: 프로젝트 일정 변경 요청의 건

김철수 팀장님께,

안녕하세요. [작성자 이름]입니다.

현재 진행 중인 프로젝트와 관련하여 일정 변경을 요청드리고자 연락드립니다.  
최근 요구사항이 추가됨에 따라 개발 범위가 확대되어, 당초 계획된 일정으로는 원활한 완료가 어려운 상황입니다.

이에 개발 일정을 2주 연장해 주실 수 있는지 검토 부탁드립니다.  
일정 조정이 승인된다면, 추가된 요구사항을 충분히 반영하여 보다 안정적으로 진행하겠습니다.

검토 부탁드리며, 가능하신 경우 회신 주시면 감사하겠습니다.

감사합니다.  
[작성자 이름] 드림


저희가 사용하는 chatgpt 같은 서비스들은 기본적으로 이전 대화 내용을 기억하지만, api를 사용할 땐 그렇지 않습니다.

그렇기 때문에 이전의 대화 내용을 기억하게 하기 위해선 이전의 대화 내용을 함께 입력 message에 담아줘야 합니다.

그냥 물어보면 이전의 대화 내용을 보지 못하기 때문에 대답하지 못함.

In [28]:
result = model.invoke([{"role": "user", "content": "내 이름이 뭐라고?"}])

print(result.content)

아직 당신 이름은 알려주지 않아서 모르겠어요.  
원하시면 이름을 알려주시면 기억해서 그렇게 부를게요.


그렇지만 이전의 대화 기록을 함께 넘겨주면 대답이 가능하다.

In [10]:
# llm이 답변한 내용을 messages에 추가
messages.append(
    {"role": "assistant", "content": result.content}
)

# 이제 앞에서 했던 대화 
user_message = "내 이름이 뭐라고?"
messages.append(
    {"role": "user", "content": user_message}
)

result = model.invoke(messages)

print(result.content)

굳건이야 😊


In [12]:
for message in messages:
    print(message)

{'role': 'system', 'content': '애교 섞인 말투로 대답해.'}
{'role': 'user', 'content': '안녕, 내 이름은 굳건이야. 앞으로 잘 부탁해.'}
{'role': 'assistant', 'content': '안녕, 굳건이! 반가워요 😊  \n앞으로 잘 부탁해요. 내가 든든하게 도와줄게요!'}
{'role': 'user', 'content': '내 이름이 뭐라고?'}


지금은 답변이 완성된 후에 한 번에 출력되지만, 생각하는 시간이 길어지거나 문장이 길어질수록 사용자 입장에선 답답하게 느껴질 수 있어요.

이럴 땐 `model.stream()`을 대신 써서 모델이 텍스트를 생성하는대로 곧바로 차례대로 출력하도록 할 수 있답니다.

In [ ]:
messages.append(
    {"role": "user", "content": "너의 이름과 너가 할 수 있는 일들에 대해 자세히 설명해줘."}
)

# chunk는 토큰 단위로 출력된다.
# end = "" 을 통해서 각 토큰을 자연스럽게 이어 붙임.
# flush = True 를 통해 바로바로 print 출력함
for chunk in model.stream(messages):
    print(chunk.text, end="", flush=True)

내 이름은 따로 없지만, 너는 나를 **챗GPT**라고 불러도 돼.  
나는 OpenAI가 만든 AI 언어 모델이야.

### 내가 할 수 있는 일
- **질문에 답하기**: 상식, 과학, 역사, 기술, 문화 등 다양한 질문에 답할 수 있어.
- **글쓰기 도와주기**: 이메일, 자기소개서, 보고서, 편지, 블로그 글 같은 걸 작성하거나 다듬어줄 수 있어.
- **번역하기**: 여러 언어를 한국어로, 또는 한국어를 다른 언어로 번역할 수 있어.
- **요약하기**: 긴 글이나 문서를 짧고 이해하기 쉽게 정리할 수 있어.
- **아이디어 내기**: 이름 짓기, 기획안, 콘텐츠 아이디어, 문장 표현 등을 같이 생각해줄 수 있어.
- **코딩 도움**: 프로그래밍 코드 작성, 설명, 디버깅, 개선 제안을 할 수 있어.
- **공부 도우미**: 개념 설명, 문제 풀이, 퀴즈, 학습 정리 등을 도와줄 수 있어.
- **대화 상대**: 그냥 이야기를 나누거나 생각을 정리하는 데도 도움을 줄 수 있어.

### 내가 조심해야 하는 점
- 나는 **실수를 할 수 있어**.
- 최신 정보나 매우 전문적인 판단은 **확인이 필요**할 수 있어.
- 의학, 법률, 금융처럼 중요한 분야는 **전문가의 조언**이 더 안전해.

원하면 다음엔  
1. 내가 어떻게 답변하는지,  
2. 내가 잘하는 일과 못하는 일,  
3. 나를 가장 잘 활용하는 방법  
이렇게 더 자세히 설명해줄게.

또는 `batch()`를 사용하여 여러 개의 메세지를 한 번에 처리하는 것도 가능합니다.

In [5]:
questions = [
    "오늘 날씨는 어때?",
    "LLM의 발전이 가져오는 이점이 뭘까?",
    "정말 AI를 모르면 앞으로 뒤쳐지게 될까?"
]

for i, response in enumerate(model.batch(questions)):
    print("Q :", questions[i])
    print("A :", response.content)

Q : 오늘 날씨는 어때?
A : 어느 지역의 날씨를 말씀하시는지 알려주시면 바로 찾아서 설명해드릴게요.  
예: **서울**, **부산**, **대구**, **인천**처럼요.
Q : LLM의 발전이 가져오는 이점이 뭘까?
A : LLM(대규모 언어 모델)의 발전이 가져오는 이점은 꽤 많습니다. 핵심만 정리하면:

1. **업무 생산성 향상**
   - 문서 작성, 요약, 번역, 이메일 초안 작성 같은 반복 작업을 빠르게 처리할 수 있습니다.
   - 개발자에게는 코드 작성, 디버깅, 테스트 작성에 도움을 줍니다.

2. **정보 접근성 향상**
   - 어려운 개념을 쉽게 설명해 주고, 필요한 정보를 빠르게 찾아 정리해 줍니다.
   - 비전문가도 기술, 법률, 의료 등 다양한 분야의 정보를 이해하기 쉬워집니다.

3. **개인 맞춤형 학습**
   - 학생이나 학습자가 자신의 수준에 맞게 질문하고 즉시 답변을 받을 수 있습니다.
   - 1:1 튜터처럼 작동해서 학습 효율을 높일 수 있습니다.

4. **창의적 작업 지원**
   - 아이디어 브레인스토밍, 글쓰기, 마케팅 카피, 콘텐츠 기획 등에 도움을 줍니다.
   - 사람이 생각의 출발점을 얻는 데 유용합니다.

5. **고객 응대 및 서비스 자동화**
   - 상담 챗봇, FAQ 응대, 예약 처리 등에서 빠르고 일관된 대응이 가능합니다.
   - 기업 입장에서는 비용 절감과 서비스 품질 향상에 도움이 됩니다.

6. **언어 장벽 완화**
   - 실시간 번역과 다국어 지원이 가능해져 글로벌 소통이 쉬워집니다.

7. **접근성 개선**
   - 시각장애인이나 읽기 어려움을 겪는 사람들에게 음성/텍스트 보조 도구로 활용될 수 있습니다.

한마디로, **LLM은 지식과 언어를 다루는 일을 더 빠르고 쉽게, 더 많은 사람이 활용할 수 있게 만드는 기술**이라고 볼 수 있습니다.

원하시면 제가 다음으로  
- **경제적 이점**,  
- **사회적 이점**,  
- **개인에게 주는 이점**  
으로 나